# DT401 Week 2 Workshop: SQL Part 7 — Interacting with SQL Databases in Python

## Learning Objectives

By the end of this notebook, you will be able to:
- Create a database engine using SQLAlchemy to connect to a SQLite database
- Load query results directly into a Pandas DataFrame using `pd.read_sql()`
- Write a DataFrame back to the database using `df.to_sql()`
- Inspect database structure programmatically using SQLAlchemy's inspector and MetaData

---

## Context Recap

The previous notebooks in this workshop used the `%%sql` magic to query the pagila database directly. That approach is convenient for exploration. In practice, though, you will often need to retrieve data from a database and then process it using Python — performing calculations, combining it with other data sources, or feeding it into a visualisation. This notebook introduces the tools that make that possible.

This is an introduction to techniques you will use regularly in DT402. The goal here is to understand what is possible and see it working — you do not need to memorise every detail.

## Part 1: Creating a Database Connection

The SQL magic commands from the earlier notebooks are a convenience layer built on top of a more fundamental Python tool: **SQLAlchemy**. SQLAlchemy provides a unified Python interface to many different SQL databases. The first step in using it is creating an **engine** — the object that manages the connection to your database.

Once you have an engine, you can use it with Pandas to read and write data, or use SQLAlchemy's own tools to inspect the database structure. The engine itself does not open a connection immediately; it creates a pool that opens connections when they are needed.

In [ ]:
import sqlalchemy as sa
import pandas as pd

### Connecting to the pagila database

For SQLite, the connection string points to the database file. The path is relative to the location of this notebook.

In [ ]:
# Connect to the pagila SQLite database
# echo=False suppresses diagnostic SQL output — set to True if you want to see what queries are being sent
engine = sa.create_engine('sqlite:///data/pagila.db', echo=False)
print("Engine created:", engine)

### Connection strings for other databases (reference only)

In your workplace, you are more likely to connect to a database on a server than a local file. The cells below show how connection strings work for other common databases. You do not need to run these — they are here for reference when you need them.

**Microsoft SQL Server — Windows Authentication:**

```python
connection_uri = sa.engine.url.URL.create(
    "mssql+pyodbc",
    host="Computer-name\\server_name",
    database="my_database",
    query={"trusted": "yes", "driver": "ODBC Driver 17 for SQL Server"}
)
engine = sa.create_engine(connection_uri)
```

**Microsoft SQL Server — SQL Authentication:**

```python
connection_uri = sa.engine.url.URL.create(
    "mssql+pyodbc",
    username="server_login",
    password="password",
    host="Computer-name\\server_name",
    database="my_database",
    query={"driver": "ODBC Driver 17 for SQL Server"}
)
engine = sa.create_engine(connection_uri)
```

**Local PostgreSQL:**

```python
connection_uri = sa.engine.url.URL.create(
    "postgresql",
    username="postgres",
    password="password",
    host="localhost",
    database="postgres"
)
engine = sa.create_engine(connection_uri)
```

The pattern is the same in each case: describe the database type, provide the credentials and location, and pass the result to `sa.create_engine()`. SQLAlchemy handles the differences between database flavours internally.

## Part 2: Reading from the Database with Pandas

Having an engine gives you a way to run queries and receive the results directly as a Pandas DataFrame — without writing the results to a file or using intermediate steps. This is where the Python-SQL bridge becomes genuinely useful: once your data is a DataFrame, every Pandas method is available to you.

**Reinforcement Resource**

[pandas.read_sql() documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_sql.html) – Official reference for the `read_sql()` function, including all parameters and examples.

*Read time: 5 minutes*

`pd.read_sql()` accepts either a table name or a full SQL query string, along with your engine. The result is a standard Pandas DataFrame.

In [ ]:
# Read the entire country table into a DataFrame
# Passing a table name rather than a query loads the full table
country_df = pd.read_sql('country', con=engine)
print(f"Shape: {country_df.shape}")
country_df.head()

You can pass any valid SQL query string, which lets you filter, join, and aggregate in the database before the data arrives in Python. This is almost always faster than loading a full table and then filtering in Pandas.

In [ ]:
# Use a SQL query to load only the first 10 rows
sample = pd.read_sql('SELECT * FROM country LIMIT 10', con=engine)
sample

For more complex analysis, you can write a multi-line query as a Python string and pass it to `read_sql()`. This keeps your SQL readable and separates the query logic from the Python logic.

In [ ]:
# Join customer and payment tables in SQL, then load the result into Python
query = """
SELECT
    c.first_name,
    c.last_name,
    SUM(p.amount) AS total_spent
FROM customer c
LEFT JOIN payment p ON c.customer_id = p.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name
ORDER BY total_spent DESC
LIMIT 10;
"""
top_customers = pd.read_sql(query, con=engine)
top_customers

Once the query result is a DataFrame, you can apply any Pandas method. The two examples below show summary statistics and row filtering — operations you would also be able to do in SQL, but which integrate naturally into a Python workflow when you are combining database data with other sources.

In [ ]:
# Load the film table and produce summary statistics for numeric columns
film_df = pd.read_sql('SELECT * FROM film', con=engine)
film_df[['length', 'rental_rate', 'replacement_cost']].describe()

In [ ]:
# Filter and sort using Pandas — equivalent to WHERE rating = 'PG' ORDER BY length DESC LIMIT 5
long_pg_films = (
    film_df[film_df['rating'] == 'PG']
    .sort_values('length', ascending=False)
    [['title', 'length', 'rating']]
    .head(5)
)
long_pg_films

> **Note:** For large datasets, it is significantly more efficient to filter and aggregate **in SQL** before loading into Pandas, rather than loading all rows and filtering in Python. SQL databases are optimised for set-based operations on large volumes of data. Use Python for operations that are more natural there — reshaping, visualisation, combining with non-SQL data sources — and let the database do what it does best.

### Practice: reading and exploring with Pandas

The two tasks below give you an opportunity to use `pd.read_sql()` with the pagila database.

**Task A:** Load the `actor` table. Using Pandas (not SQL), find the 5 actors with the longest last names.

In [ ]:
# Task A: Load the actor table and find the 5 actors with the longest last names


<details>
<summary>Solution</summary>

```python
actor_df = pd.read_sql('actor', con=engine)
actor_df['name_length'] = actor_df['last_name'].str.len()
actor_df.sort_values('name_length', ascending=False)[['first_name', 'last_name', 'name_length']].head(5)
```

The key step is using `.str.len()` to calculate the length of each last name as a new column, then sorting by that column. You could also do this in SQL with `LENGTH(last_name)` — try it both ways and compare.

</details>

**Task B:** Write a SQL query to find the total payments received by each store. Load the result into a DataFrame and display it.

In [ ]:
# Task B: Total payments by store — write a SQL query, load with pd.read_sql(), and display


<details>
<summary>Solution</summary>

```python
store_payments_query = """
SELECT
    s.store_id,
    SUM(p.amount) AS total_payments
FROM payment p
LEFT JOIN staff st ON p.staff_id = st.staff_id
LEFT JOIN store s ON st.store_id = s.store_id
GROUP BY s.store_id
ORDER BY total_payments DESC;
"""
store_payments = pd.read_sql(store_payments_query, con=engine)
store_payments
```

This query traces payments through the `staff` table to reach `store` — staff are assigned to stores, and payments are assigned to staff. This is a pattern you will encounter frequently: reaching a piece of information through an intermediate table.

</details>

## Part 3: Writing to the Database with Pandas

Reading from a database is the most common operation, but there are situations where you need to write data back — for instance, storing the results of a Python analysis, creating a staging table for further processing, or persisting a filtered dataset. `DataFrame.to_sql()` handles this.

**Reinforcement Resource**

[pandas.DataFrame.to_sql() documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html) – Official reference for writing DataFrames to SQL databases.

*Read time: 5 minutes*

The `if_exists` parameter controls what happens if the target table already exists:
- `'fail'` — raise an error (the default — prevents accidental overwrites)
- `'replace'` — drop the existing table and create a new one
- `'append'` — add the new rows to the existing table

The example below demonstrates the full write-verify-clean-up cycle.

In [ ]:
# Create a small DataFrame to write — using 5 rows from the country table
test_data = pd.read_sql('SELECT * FROM country LIMIT 5', con=engine)
print("Data to write:")
print(test_data)

In [ ]:
# Write the DataFrame to a new table called 'dummy_table'
# if_exists='replace' means the table is dropped and recreated if it already exists
test_data.to_sql('dummy_table', engine, if_exists='replace', index=False)
print("Table written successfully")

In [ ]:
# Read it back to confirm the write was successful
pd.read_sql('dummy_table', con=engine)

In [ ]:
# Clean up — remove the test table once we are done
# Using a context manager ensures the connection is properly closed after the operation
with engine.connect() as conn:
    conn.execute(sa.text('DROP TABLE IF EXISTS dummy_table'))
    conn.commit()
print("Cleaned up")

The clean-up cell above uses a **context manager** (`with engine.connect() as conn:`) to manage the connection explicitly. This pattern ensures the connection is released as soon as the block finishes, even if an error occurs. It is the recommended approach whenever you are executing SQL statements directly rather than using `pd.read_sql()` or `df.to_sql()`.

## Part 4: Inspecting the Database Structure

When you connect to an unfamiliar database in a real project, you may not have an ERD to hand. SQLAlchemy's inspection tools let you explore the database structure programmatically — finding out what tables exist and what columns they contain without writing raw SQL.

This complements the SQL-based approach you saw in the [SQLite Table Identification](SQLite_table_identification.ipynb) notebook, which uses `sqlite_master` and `pragma_table_info` for the same purpose. Both approaches are useful; the SQLAlchemy approach works across different database types, not just SQLite.

**Reinforcement Resource**

[SQLAlchemy inspection documentation](https://docs.sqlalchemy.org/en/20/core/reflection.html) – Official reference for database reflection and the Inspector API.

*Read time: 5 minutes*

In [ ]:
# Get a list of all table names in the database
inspector = sa.inspect(engine)
inspector.get_table_names()

In [ ]:
# Get detailed column information for a specific table
columns = inspector.get_columns('customer')
for col in columns:
    print(f"{col['name']:20} {str(col['type']):20} nullable={col['nullable']}")

For a broader view — all tables and all their columns at once — use `MetaData` with `reflect()`. This loads the full database schema into memory, which you can then iterate over.

In [ ]:
# Use MetaData to print all tables and their column names
metadata = sa.MetaData()
metadata.reflect(engine)

for name, table in metadata.tables.items():
    print(f"{name}: {list(table.columns.keys())}")

> **What to observe:** Run the cell above and count how many tables the pagila database contains. Which table has the most columns? You can cross-check this against the ERD notebook to confirm your answer.

<details>
<summary>Solution</summary>

The pagila database contains 14 tables. The `film` table has the most columns — it stores detailed information about each title including rating, length, rental rate, replacement cost, language, and several feature flags (`has_trailers`, `has_commentaries`, etc.). This makes sense for the domain: films have many attributes that need to be stored, whereas a table like `language` needs only an id and a name.

</details>

## Part 5: Closing the Connection

When you have finished working with a database, you should dispose of the engine. This releases the connections back to the pool and prevents them from being held open unnecessarily. In a Jupyter notebook, the kernel will handle cleanup when it shuts down — but calling `dispose()` explicitly is good practice, particularly when you write scripts or applications that run outside a notebook environment.

In [ ]:
engine.dispose()
print("Connection closed")

---

## Reflection

**Write your reflections in the empty cell below.**

1. **Workplace Application:** Think about a database or data source you interact with in your organisation. How do you currently extract data from it? Could the `pd.read_sql()` approach connect to that system? What would change about your workflow if it could?

2. **SQL vs Python:** Several of the examples in this notebook showed the same operation performed in SQL (before loading) and in Pandas (after loading). What factors would help you decide which approach to use in a real project?

3. **Next Steps:** This notebook introduced SQLAlchemy and `pd.read_sql()` at a basic level. What aspect of connecting Python to a database would be most useful to explore further in your current role?

1. 

2.

3.


---

## Summary

**You have completed SQL Part 7: Interacting with SQL Databases in Python.** You can now:
- Create a SQLAlchemy engine to connect to a SQLite database (and recognise the connection string pattern for other databases)
- Load query results into a Pandas DataFrame using `pd.read_sql()`
- Write a DataFrame to the database using `df.to_sql()`
- Inspect database structure programmatically using `sa.inspect()` and `sa.MetaData()`

**Next:** The skills introduced here — particularly `pd.read_sql()` — form the foundation for database integration work in DT402, where you will build more substantial data pipelines and applications that connect Python to live database systems.

**Action items:**
1. Save this notebook (Ctrl+S)
2. Commit to Git: `git commit -m "Complete Week 2 Workshop SQL Part 7 Python database interaction"`
3. Record your reflections in your professional logbook

---

| Tool | Purpose |
|:-----|:--------|
| `sa.create_engine(url)` | Create a connection pool for a database |
| `pd.read_sql(query, con=engine)` | Run SQL and return a Pandas DataFrame |
| `df.to_sql(name, engine)` | Write a DataFrame to a database table |
| `sa.inspect(engine)` | Inspect tables and columns in the database |
| `sa.MetaData()` + `.reflect(engine)` | Load full table and column metadata |
| `engine.dispose()` | Release all database connections |

**See also:**
- [SQLAlchemy documentation](https://docs.sqlalchemy.org/)
- [pandas.read_sql() documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_sql.html)
- [pandas.DataFrame.to_sql() documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html)
- [ERD — Database Schema](ERD.ipynb)
- [SQLite Table Identification](SQLite_table_identification.ipynb)